# Machine Learning: Exercise session 05

In this exercise session we explore LDA and QDA.

First, we generate and analyse synthetic datasets with different characteristics.
Visualizing the fitted models illustrates the differences between LDA and QDA.

Then, we analyse the [MNIST dataset](https://en.wikipedia.org/wiki/MNIST_database) using LDA and QDA models.

Lastly, we show for the one-dimensional case that in the the QDA classifier is a quadratic function of the predictor values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes

## Problem 1 - Synthetic Data

Modified example from https://scikit-learn.org/stable/modules/lda_qda.html


### Data generation

Since LDA and QDA are generative models,
we can generate synthetic data that perfectly fits the assumptions of LDA and QDA.
Complete the function below to generate a two-class dataset, split into training and test sets of indicated sizes, that fits the assumptions of LDA and QDA.

In [ ]:
def make_two_class_dataset(
    n_train=100,
    n_test=1000,
    p_0=0.5,
    mean_0=(0, 0),
    mean_1=(1, 0),
    cov_0=np.eye(2),
    cov_1=np.eye(2),
    seed=42,
):
    # Set the seed for reproducibility
    np.random.seed(seed)

    # Total number of samples
    n_samples = ...

    # Generate y
    y = ...

    # Create an empty feature matrix (zeros as placeholders)
    X = ...

    # Sample the features based on class labels
    ...

    # Split the data into training and test sets
    ...

    return X_train, X_test, y_train, y_test

To see what the generated data looks like, complete the `plot_data` function below and call it with a dataset generated by your `make_two_class_dataset` function.
We use the object-based interface of `matplotlib` here, which makes it easier to combine multiple plots into one figure.
See the [documentation](https://matplotlib.org/stable/users/explain/figure/api_interfaces.html#api-interfaces) for details.

In [ ]:
def plot_data(X, y, ax: Axes):
    # Plot a scatter plot of the data points
    # Class 0 in red, class 1 in blue
    ax.scatter(...)
    ax.scatter(...)

    # Set equal aspect ratio (makes circles look like circles etc.)
    ax.set_box_aspect(1)

In [ ]:
# Test the data generation and plotting
# Not specifying any arguments uses the default parameters
X_train, X_test, y_train, y_test = make_two_class_dataset()

# We split the plot into two subplots, one for training and one for test data
# Each subplot can be addressed via the axs array
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
plot_data(X_train, y_train, axs[0])
axs[0].set_title("Training Data")
plot_data(X_test, y_test, axs[1])
axs[1].set_title("Test Data")
plt.show()

### Model fitting and evaluation

Fit an LDA and a QDA model to the training data and evaluate their accuracy on the test data.
Since we want to repeat this check for multiple different datasets, complete and call the `print_scores` function below.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Specify `store_covariance=True` in the constructor to be able to analyze the estimated covariance matrices later
lda = ...

qda = ...

In [ ]:
def print_scores(estimator, X_test, y_test):
    if isinstance(estimator, LinearDiscriminantAnalysis):
        model_name = ...
    elif isinstance(estimator, QuadraticDiscriminantAnalysis):
        model_name = ...
    else:
        model_name = "???"
    X_0 = X_test[y_test == 0]
    X_1 = ...
    y_0 = ...
    y_1 = ...
    print(f"Test scores of {...}:")
    print(f"  on class 0: {...}")
    print(f"  on class 1: {...}")
    print(f"  overall:    {...}")
    print()

In [ ]:
print_scores(lda, X_test, y_test)
print_scores(qda, X_test, y_test)

### Model visualization

In order to visualize the fitted models, complete the `plot_results` function below.
The functions `plot_ellipse` and `plot_decision_boundary` do what their names suggest and can be used as given below.

In [ ]:
from matplotlib.patches import Ellipse
def plot_ellipse(mean, cov, color, ax: Axes):
    # Represents a mean+covariance as an ellipse
    v, w = np.linalg.eigh(cov)
    u = w[0] / np.linalg.norm(w[0])
    angle = np.arctan(u[1] / u[0])
    angle = 180 * angle / np.pi  # convert to degrees
    ell = Ellipse(
        mean,
        2 * v[0] ** 0.5,
        2 * v[1] ** 0.5,
        angle=180 + angle,
        facecolor=color,
    )
    ell.set_clip_box(ax.bbox)
    ell.set_alpha(0.4)
    ax.add_artist(ell)

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
def plot_decision_boundary(estimator, X, ax: Axes):
    DecisionBoundaryDisplay.from_estimator(
        estimator,
        X,
        response_method="predict_proba",
        plot_method="pcolormesh",
        ax=ax,
        cmap="RdBu",
        alpha=0.3,
    )
    DecisionBoundaryDisplay.from_estimator(
        estimator,
        X,
        response_method="predict_proba",
        plot_method="contour",
        ax=ax,
        alpha=1.0,
        levels=[0.5],
    )

In [ ]:
def plot_results(estimator, X, y, ax: Axes):
    class_colors = ["red", "blue"]
    plot_decision_boundary(estimator, X, ax)
    y_pred = estimator.predict(X)
    for i in range(2):
        # Compute the indices of class i samples that are predicted (in)correctly
        ind_wrong = ...
        ind_right = ...

        # Make a scatter plot of the points
        # Use marker '.' for correctly classified points
        ax.scatter(..., c=class_colors[i])
        # Use marker 'x' for incorrectly classified points
        ax.scatter(..., c=class_colors[i])

    # Add ellipses for the estimated class means and covariances
    if isinstance(estimator, LinearDiscriminantAnalysis):
        covariances = [estimator.covariance_] * 2  # same covariance for both classes
    else:
        covariances = estimator.covariance_
    means = np.array(estimator.means_)
    plot_ellipse(means[0], covariances[0], class_colors[0], ax)
    plot_ellipse(means[1], covariances[1], class_colors[1], ax)

    # Plot the estimated class means as black points
    ...

    ax.set_box_aspect(1)

Test your function by calling it with the training data you generated above.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
plot_results(lda, X_train, y_train, axs[0])
axs[0].set_title("LDA")
plot_results(qda, X_train, y_train, axs[1])
axs[1].set_title("QDA")
plt.show()

### Combined analysis

Lastly, we put the model fitting, plotting, and score computation together in an `analyze_dataset` function that takes care of everything.

In [ ]:
def analyze_dataset(X_train, X_test, y_train, y_test ):
    # Fit estimators on training data
    lda = ...

    qda = ...

    # Plot Training data, LDA results, and QDA results
    _, axs = plt.subplots(1, 3, figsize=(10, 4), sharex=True, sharey=True)
    plot_data(...)
    axs[0].set_title(...)

    plot_results(...)
    axs[1].set_title(...)

    plot_results(...)
    axs[2].set_title(...)

    plt.show()

    # Compute and print scores on test data
    print_scores(...)
    print_scores(...)


In [ ]:
# Create dataset and analyze it
X_train, X_test, y_train, y_test = make_two_class_dataset()
analyze_dataset(X_train, X_test, y_train, y_test)

In [ ]:
# Shorter form using argument unpacking
# Warning: not very readable, only use when you are sure about arguments/returns!
data = make_two_class_dataset()
analyze_dataset(*data)

# or even:
# analyze_dataset(*make_two_class_dataset())

Using this "pipeline", we can now now generate and visualize LDA/QDA fits for different datasets.

In [ ]:
# Example: same covariance, easy to separate, plenty of data
X_train, X_test, y_train, y_test = make_two_class_dataset(
    n_train=300,
    mean_1=(3, 0),
)
analyze_dataset(X_train, X_test, y_train, y_test)

In [ ]:
# Example: same covariance, hard to separate, plenty of data
X_train, X_test, y_train, y_test = make_two_class_dataset(
    n_train=1000,
    mean_1=(0.5, 0),
)
analyze_dataset(X_train, X_test, y_train, y_test)

In [ ]:
# Example: hard to separate, little data
X_train, X_test, y_train, y_test = make_two_class_dataset(
    n_train=20,
    mean_1=(1, 0),
)
analyze_dataset(X_train, X_test, y_train, y_test)

In [ ]:
# Example: different covariances, easy to separate
cov_0 = np.array([[1.75, 1.25], [1.25, 1.75]])
cov_1 = np.array([[3, 0], [0, 0.1]])
X_train, X_test, y_train, y_test = make_two_class_dataset(
    n_train=100,
    mean_1=(3, 0),
    cov_0=cov_0,
    cov_1=cov_1,
)
analyze_dataset(X_train, X_test, y_train, y_test)

In [ ]:
# Tricky example: different covariances, same mean
cov_0 = np.array([[3, 2.9], [2.9, 3]])
cov_1 = np.array([[3, -2.9], [-2.9, 3]])
X_train, X_test, y_train, y_test = make_two_class_dataset(
    n_train=1000,
    mean_1=(0, 0),
    cov_0=cov_0,
    cov_1=cov_1,
)
analyze_dataset(X_train, X_test, y_train, y_test)

In [ ]:
# Example: small cluster vs. large cluster
cov_0 = np.array([[0.1, 0], [0, 0.1]])
cov_1 = np.array([[5, 0], [0, 5]])
X_train, X_test, y_train, y_test = make_two_class_dataset(
    n_train=100,
    p_0=0.3,
    mean_1=(0, 2),
    cov_0=cov_0,
    cov_1=cov_1,
)
analyze_dataset(X_train, X_test, y_train, y_test)

## Problem 2 - MNIST Data

### Setup and exploration

In [ ]:
# Import package/module for data
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Import modules for feature engineering and modelling
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)

from sklearn.metrics import accuracy_score

Import the dataset `digits.csv`, separate the predictors and the target variable and split it into training and test set. When you perform the splitting, set `random_state=40` and `test_size=50`.

In [ ]:
dat = ...

# Define X and Y
X = ...
y = ...

In [ ]:
# Split dataset into train-test
X_train, X_test, y_train, y_test = train_test_split(...)

In [ ]:
# Convert to numpy arrays (easier to plot etc.)
X_train = np.array(X_train)
y_train = np.array(y_train).ravel()
X_test = np.array(X_test)
y_test = np.array(y_test).ravel()

We now want to plot some of the observations of the dataset. Each row of the predictor matrix is 256 vector corresponding to a 16 x 16 image.

In [ ]:
# Print the image as a 16x16 array
np.random.seed(42)
row_index = np.random.randint(0, len(X_train))
np.set_printoptions(linewidth = 150)
print(X_train[...].reshape(...))

In [ ]:
# Better: plot the array as an image
np.random.seed(42)
row_index = np.random.randint(0, len(X_train))
image_data = ...
plt.imshow(image_data, cmap="binary")
plt.title(f"Class = {...}")
plt.show()

In [ ]:
# Plot some figures
n_rows = 4
n_cols = 4
fig, ax = plt.subplots(n_rows, n_cols, figsize=(8, 8))
np.random.seed(42)
for i in range(n_rows):
    for j in range(n_cols):
        row_index = ...
        image_data = ...
        ax[i, j].imshow(...)
        ax[i, j].set_title(...)
        ax[i, j].set_xticks([])
        ax[i, j].set_yticks([])
plt.show()

### LDA

We now try to fit an LDA model.
Mathematically, LDA is invariant under linear transformations (rescaling) of the predictors.
However, for numerical stability or when using regularization, it is a good idea to rescale the data anyways.
Create a `Pipeline` that first standardizes the data and then fits an LDA model.

In [ ]:
# Define pipeline
lda = LinearDiscriminantAnalysis(shrinkage = 'auto', solver = 'lsqr')
scaler_lda = StandardScaler()
pipe_lda = Pipeline([
    ...
])

# Fit pipeline
...

# Predict on training data
...
print("LDA --- Accuracy on training data:", ...)

# Predict on test data
...
print("LDA --- Accuracy on test data:", ...)


Since LDA estimates the class-conditional means, it can be useful to try to understand qualitatively how these look like. What do these mean correspond to?

In [ ]:
...

Since we know that the data represents images of digits, we can try to plot the means as images.

In [ ]:
def plot_means(means, method_name):
    plt.figure(figsize=(10, 5))
    for i in range(10):
        l1_plot = plt.subplot(2, 5, i + 1)
        l1_plot.imshow(...)
        l1_plot.set_xticks(())
        l1_plot.set_yticks(())
        l1_plot.set_xlabel(...)
    plt.suptitle("Means estimated by " + method_name)
    plt.show()

In [ ]:
plot_means(lda.means_, "LDA")

Plotting the means, we see that they roughly look like the digits they are supposed to represent.
However, they are a bit distorted. Why is that and how could we fix that?

*Hint: Remember that we fitted an entire pipeline, not just the LDA estimator*

In [ ]:
...

plot_means(lda_means, "LDA")

### QDA

Repeat the same analysis as above but now using QDA.
If you create a `QuadraticDiscriminantAnalysis` object without any arguments, you will see that it does not work properly.
Why is that?
Fix the problem by setting the argument `reg_param` to a small value (e.g. 0.01).

In [ ]:
# QDA
# Define pipeline
...

# Fit pipeline
...

In [ ]:
# Predict on training data
...
print("QDA --- Accuracy on training data:", ...)

# Predict on test data
...
print("QDA --- Accuracy on test data:", ...)

As we did for LDA, plot the means estimated by QDA. Call the function you created by passing as an argument the matrix with the estimated means.

In [ ]:
# Plot means
...

Do these look different from the means estimated by LDA? Why (not)?
Verify your answer using `numpy`.

In [ ]:
...

_Bonus_: What do the estimated covariance matrices look like, what do they represent? Can you think of a way to visualize them?

## Problem 3

This problem is about quadratic discriminant analysis (QDA).
The observations within each class are modeled by a normal distribution with a class specific mean vector and a class specific covariance matrix. We consider the simple case where $p = 1$, (i.e., there is only one feature) and that we have $q = 2$ classes. Thus, for $j=0,1$, the conditional distribution $X\mid Y=j$ of the $j$-th class is one-dimensional normal $N(\mu_j, \sigma^2_j)$. 
Show that in this case, the QDA classifier is not linear in $x$, but _quadratic_,
i.e., that the decision boundary is given as the solution of a quadratic equation in $x$.